# 02b — UC2 FedAvg on the new (label-skew) partitions

Trains FedAvg across the α sweep on the **new** partitions from notebook 00,
for **both** evaluation designs:

- **Option A — client-local** (`new_partitions/`): each client tested on its own
  load regime. Heterogeneity shows up in **per-client MAE spread**.
- **Option B — global pool** (`new_partitions_global/`): every client tested on
  the same 3000-sample pool. Measures the **global objective**; read it as the
  **gap to Centralized**, not the raw level.

Switch designs with `PARTITION_VARIANT` in the config cell and re-run.
Results are written to `results/newpart/` (Option A) or `results/newpart_global/`
(Option B), so nothing overwrites your original AP-ID results.

> **Heads-up from the notebook-00 data:** on this dataset the heterogeneity is
> dominated by **quantity skew** (per-user sample-count Gini 0.55 → 0.06 across α)
> while **label skew is mild** (y-mean Gini ≈ 0.03–0.07). Expect modest movement
> in the *global mean* MAE across α and a clearer signal in the *per-client spread*.


In [ ]:
import sys, os
sys.path.append("..")
import UC2Utils as uc2
sys.path.insert(0, uc2.LIB_DIR)

import numpy as np
import matplotlib.pyplot as plt
import pickle, json
import torch

print(f"Device: {uc2.DEFAULT_CONFIG['device']}")
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0),
          f"| free {torch.cuda.mem_get_info()[0]/1024**3:.1f} GB")

## Configuration

In [ ]:
ALPHAS = [0.01, 0.1, 0.5, 1.0, 5.0, 10.0]
MODEL  = "lstm"

# Which evaluation design to run:
#   "client_local" -> new_partitions/        (Option A)
#   "global"       -> new_partitions_global/ (Option B)
PARTITION_VARIANT = "client_local"

# First run on a given variant: set True so it TRAINS instead of loading old pkls.
RETRAIN = False

OVERRIDES = dict(
    num_glob_iters=300,
    local_epochs=20,
    num_users=20,
    early_stopping_patience=25,
    batch_size=32,
)

## Partition wiring

Generalises the toggle from notebook 02 to pick the partition **variant** and to
route results to a per-variant results folder. `run_experiment` is otherwise
unchanged — only `dataset_path` and `result_path` are redirected.

In [ ]:
# config: folder names must match what notebook 00 wrote
VARIANT_SUBDIR = {"client_local": "new_partitions",
                  "global":       "new_partitions_global"}
VARIANT_RESULTS = {"client_local": "newpart",
                   "global":       "newpart_global"}
assert PARTITION_VARIANT in VARIANT_SUBDIR, PARTITION_VARIANT

_NP_LOOKBACK = uc2.DEFAULT_CONFIG["lookback"]   # 60
_NP_STEPS    = uc2.DEFAULT_CONFIG["steps"]      # 1
_NP_SUBDIR   = VARIANT_SUBDIR[PARTITION_VARIANT]
_NP_RESULTS  = VARIANT_RESULTS[PARTITION_VARIANT]

# ── pristine originals (idempotent — fixes the re-run RecursionError) ─────────
# Bug it prevents: re-running this cell used to do `_orig_make_args = uc2.make_args`
# AFTER uc2.make_args had already been patched, so the patch called itself forever.
# Fix: if uc2.* is currently patched (its functions live in __main__, not UC2Utils),
# reload UC2Utils to restore the real module functions BEFORE capturing them.
import importlib, sys
if getattr(uc2.make_args, "__module__", "") != uc2.__name__:
    importlib.reload(uc2)
    sys.path.insert(0, uc2.LIB_DIR)
_orig_make_args     = uc2.make_args
_orig_result_exists = uc2.result_exists
_orig_load_result   = uc2.load_result
assert _orig_make_args.__module__ == uc2.__name__, \
    "make_args is still patched -- restart the kernel and re-run."

_NEWPART_ON = {"flag": False}

def _newpart_dataset_path():
    return os.path.join(uc2.DATA_PART, _NP_SUBDIR,
                        f"lookback_{_NP_LOOKBACK}", f"steps_{_NP_STEPS}")

def _redirect_result_path(result_path, algorithm, alpha, model, seed=0):
    base = os.path.join(uc2.RESULTS, _NP_RESULTS)
    if result_path is None:
        return os.path.join(base, algorithm.lower(), f"alpha_{alpha}", model, f"rep_{seed}")
    rel = os.path.relpath(os.path.abspath(result_path), os.path.abspath(uc2.RESULTS))
    return os.path.join(base, rel)

def _patched_make_args(algorithm, alpha, result_path=None, seed=0, **overrides):
    args = _orig_make_args(algorithm, alpha, result_path=result_path, seed=seed, **overrides)
    if _NEWPART_ON["flag"]:
        args.dataset_path = _newpart_dataset_path()
        model = {**uc2.DEFAULT_CONFIG, **overrides}["model"]
        new_rp = _redirect_result_path(result_path, algorithm, alpha, model, seed=seed)
        new_rp = os.path.relpath(new_rp)          # lib dislikes spaces in abs paths
        os.makedirs(new_rp, exist_ok=True)
        args.result_path = new_rp
    return args

def _patched_result_exists(algorithm, alpha, model="lstm", seed=0):
    if _NEWPART_ON["flag"]:
        p = os.path.join(uc2.RESULTS, _NP_RESULTS, algorithm.lower(),
                         f"alpha_{alpha}", model, f"rep_{seed}", "full_results.pkl")
        return os.path.exists(p)
    return _orig_result_exists(algorithm, alpha, model, seed=seed)

def _patched_load_result(algorithm, alpha, model="lstm", seed=0):
    if _NEWPART_ON["flag"]:
        p = os.path.join(uc2.RESULTS, _NP_RESULTS, algorithm.lower(),
                         f"alpha_{alpha}", model, f"rep_{seed}", "full_results.pkl")
        with open(p, "rb") as f:
            return pickle.load(f)
    return _orig_load_result(algorithm, alpha, model, seed=seed)

def use_new_partitions(on=True):
    """Toggle the selected new-partition variant for subsequent run_experiment calls."""
    _NEWPART_ON["flag"] = bool(on)
    uc2.make_args     = _patched_make_args     if on else _orig_make_args
    uc2.result_exists = _patched_result_exists if on else _orig_result_exists
    uc2.load_result   = _patched_load_result   if on else _orig_load_result
    state = f"NEW [{PARTITION_VARIANT}]" if on else "ORIGINAL AP-ID"
    print(f"[wiring] partitions = {state}  (orig from {_orig_make_args.__module__})")
    if on:
        dp = _newpart_dataset_path()
        print(f"[wiring] dataset_path -> {dp}")
        print(f"[wiring] results      -> {os.path.join(uc2.RESULTS, _NP_RESULTS, '...')}")
        miss = [a for a in ALPHAS if not os.path.exists(
            os.path.join(dp, f"u{uc2.DEFAULT_CONFIG['n_users']}-alpha{a}-ratio1",
                         "train", "train.pt"))]
        if miss:
            print(f"[wiring] !! MISSING partitions for alpha={miss} -- "
                  f"run notebook 00 generation for variant '{PARTITION_VARIANT}' first!")

use_new_partitions(True)


In [ ]:
print("make_args is patched:", uc2.make_args is _patched_make_args)
print("result_exists is patched:", uc2.result_exists is _patched_result_exists)
print("flag:", _NEWPART_ON["flag"])

In [ ]:
# --- VERIFY: client-local partition wired, clients get distinct test sets ---
import torch
_args = uc2.make_args("FedAvg", ALPHAS[0], **OVERRIDES)
print("dataset_path =", _args.dataset_path)
assert "new_partitions" in _args.dataset_path and "global" not in _args.dataset_path, \
    "WIRING OFF: dataset_path is not the client-local new_partitions root."
_server = uc2.create_server(_args)
print(f"\n{'client':>8}{'test n':>10}{'target mean':>14}")
means = []
for c in _server.users[:5]:
    xb, yb = next(iter(c.testloaderfull))
    means.append(round(float(yb.float().mean()), 4))
    print(f"{str(c.id):>8}{c.test_samples:>10}{yb.float().mean():>14.4f}")
assert len(set(means)) > 1, "Clients have identical test means -> still shared pool."
print("\n[OK] Distinct per-client test sets. Client-local wiring confirmed.")

## Train FedAvg for each α

On the **first** run of a variant keep `RETRAIN=True` so it trains rather than
loading stale pickles. Subsequent runs can set `RETRAIN=False` to just reload.

In [ ]:
print("make_args patched:", uc2.make_args is _patched_make_args)   # must be True
print("result_exists patched:", uc2.result_exists is _patched_result_exists)  # must be True

In [ ]:
import inspect
print(inspect.signature(uc2.result_exists))   # want: (algorithm, alpha, model='lstm', seed=0)
print(inspect.signature(uc2.make_args))        # want: (algorithm, alpha, result_path=None, seed=0, **overrides)

In [ ]:
import inspect
print("orig is real module func:", uc2._orig_make_args if False else None)  # ignore
print("orig signature:", inspect.signature(_orig_make_args))   # want: (algorithm, alpha, result_path=None, seed=0, **overrides)
print("orig source file:", inspect.getsourcefile(_orig_make_args))  # want: .../UC2Utils.py  NOT /tmp/ipykernel...

In [ ]:
SEEDS = [0,42, 123]          # seed 0 already on disk; not in this list, so never touched

results_fedavg = {}        # keyed (alpha, seed)
for seed in SEEDS:
    for alpha in ALPHAS:
        print(f"\n{'='*60}\n  FedAvg α={alpha} seed={seed}\n{'='*60}")
        if not RETRAIN and uc2.result_exists("FedAvg", alpha, seed=seed):
            print("  [done] loading cached result.")
            results_fedavg[(alpha, seed)] = uc2.load_result("FedAvg", alpha, seed=seed)
            continue
        try:
            server, result = uc2.run_experiment(
                algorithm="FedAvg", alpha=alpha, seed=seed, **OVERRIDES)
            results_fedavg[(alpha, seed)] = result
        except Exception as e:
            import traceback; traceback.print_exc()

In [ ]:
# ── Seed-aware summary: confirms all (alpha, seed) ran, reports best scaled MAE ──
import numpy as np

SEEDS_ALL = [0, 42, 123]   # include seed 0 (already on disk) for the bands
ALPHAS_SORTED = sorted(ALPHAS)

def _best_scaled(r):
    glob = r["metrics"].get("glob_test_metric", [])
    if not glob: return float("nan")
    return min(m.get("mae", float("inf")) for m in glob)   # SCALED, lower=better

# pull every (alpha, seed) from disk so seed 0 is included even though the loop skipped it
cube = {}   # (alpha, seed) -> best scaled MAE
for a in ALPHAS_SORTED:
    for s in SEEDS_ALL:
        if uc2.result_exists("FedAvg", a, seed=s):
            cube[(a, s)] = _best_scaled(uc2.load_result("FedAvg", a, seed=s))

# coverage report
print("Coverage (✓ = result on disk):")
print(f"{'α':>8}" + "".join(f"{'s'+str(s):>8}" for s in SEEDS_ALL))
for a in ALPHAS_SORTED:
    row = "".join(f"{'✓' if (a,s) in cube else '—':>8}" for s in SEEDS_ALL)
    print(f"{a:>8}{row}")

# mean ± std across seeds
print(f"\n{'α':>8}{'mean MAE':>11}{'std':>9}{'n':>4}   per-seed")
for a in ALPHAS_SORTED:
    vals = np.array([cube[(a,s)] for s in SEEDS_ALL if (a,s) in cube])
    if vals.size:
        seeds_str = ", ".join(f"{cube[(a,s)]:.4f}" for s in SEEDS_ALL if (a,s) in cube)
        std = vals.std(ddof=1) if vals.size > 1 else 0.0
        print(f"{a:>8}{vals.mean():>11.4f}{std:>9.4f}{vals.size:>4}   [{seeds_str}]")
    else:
        print(f"{a:>8}{'—':>11}{'—':>9}{0:>4}")

In [ ]:
# ── FedAvg convergence, mean across seeds with min–max band, per alpha ──
uc2.setup_plot_style()
fig, ax = plt.subplots(figsize=(11, 6))
cmap = plt.cm.viridis(np.linspace(0, 0.9, len(ALPHAS_SORTED)))

for a, col in zip(ALPHAS_SORTED, cmap):
    curves = []
    for s in SEEDS_ALL:
        if not uc2.result_exists("FedAvg", a, seed=s): continue
        r = uc2.load_result("FedAvg", a, seed=s)
        curves.append([m["mae"] for m in r["metrics"].get("glob_test_metric", [])])
    if not curves: continue
    L = min(len(c) for c in curves)               # align to shortest (early-stop differs)
    M = np.array([c[:L] for c in curves])
    mean = M.mean(0)
    ax.plot(mean, color=col, lw=2, label=f"α={a} (n={len(curves)})")
    if M.shape[0] > 1:
        ax.fill_between(range(L), M.min(0), M.max(0), color=col, alpha=0.15)

ax.set_xlabel("Communication round")
ax.set_ylabel("Scaled MAE")          # unitless
ax.set_title(f"FedAvg convergence (mean ± seed range) — {PARTITION_VARIANT}")
ax.legend(ncol=2, fontsize=9); plt.tight_layout(); plt.show()

In [ ]:
# ── Final scaled MAE vs α, mean ± std over seeds (the α-sensitivity figure) ──
uc2.setup_plot_style()
fig, ax = plt.subplots(figsize=(10, 5.5))

means, stds, xs = [], [], []
for a in ALPHAS_SORTED:
    vals = np.array([cube[(a,s)] for s in SEEDS_ALL if (a,s) in cube])
    if vals.size:
        xs.append(a); means.append(vals.mean())
        stds.append(vals.std(ddof=1) if vals.size > 1 else 0.0)

ax.errorbar(xs, means, yerr=stds, marker="o", lw=2, capsize=4,
            color=uc2.COLORS["FedAvg"], label="FedAvg (mean ± std)")
ax.set_xscale("log")
ax.set_xlabel("α  (log scale; left = more heterogeneous)")
ax.set_ylabel("Best scaled MAE")
ax.set_title(f"FedAvg α-sensitivity across seeds — {PARTITION_VARIANT}")
ax.legend(); plt.tight_layout(); plt.show()

print("If the error bars are tight and overlap across α → no real heterogeneity")
print("effect (consistent with the mild-heterogeneity finding). The α=0.01 point")
print("is the one to watch: if its band sits clearly below the rest AND is narrow,")
print("the dip is a stable partition artifact, not seed noise.")

## Training convergence per α

In [ ]:
uc2.setup_plot_style()
fig, ax = plt.subplots(figsize=(10, 5))
for alpha, r in sorted(results_fedavg.items()):
    maes = [m["mae"] for m in r["metrics"].get("glob_test_metric", [])]
    ax.plot(maes, label=f"α={alpha}", lw=2)
ax.set_ylim(0.25, 0.40)          # scaled MAE actually lives here
ax.set_xlabel("Communication round")
ax.set_ylabel("Scaled MAE")       # unitless, NOT MB
ax.set_title(f"FedAvg convergence — {PARTITION_VARIANT}")
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
uc2.setup_plot_style()
fig, ax = plt.subplots(figsize=(10, 5))
for alpha, r in sorted(results_fedavg.items()):
    maes = [m["unscaled_mae"] for m in r["metrics"].get("glob_test_metric", [])]
    ax.plot(maes, label=f"α={alpha}", lw=2)
ax.set_ylim(0, 100)          # scaled MAE actually lives here
ax.set_xlabel("Communication round")
ax.set_ylabel("Unscaled MAE")       # unitless, NOT MB
ax.set_title(f"FedAvg convergence — {PARTITION_VARIANT}")
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
import numpy as np
r = results_fedavg[(0.01, 42)]
g = r["metrics"]["glob_test_metric"]
sc = np.array([m["mae"] for m in g])
un = np.array([m["unscaled_mae"] for m in g])
print(f"scaled  : first={sc[0]:.5f}  last={sc[-1]:.5f}  Δ={sc[0]-sc[-1]:+.5f}  ({100*(sc[0]-sc[-1])/sc[0]:+.1f}%)")
print(f"unscaled: first={un[0]:.5f}  last={un[-1]:.5f}  Δ={un[0]-un[-1]:+.5f}  ({100*(un[0]-un[-1])/un[0]:+.1f}%)")

In [ ]:
import numpy as np
r = results_fedavg[(0.01, 42)]
pm = np.array(r["per_user"]["metrics"]["unscaled_mae"])
ns = np.array(r["per_user"]["num_samples"])
# if your per_user block also stores per-client target means, sort by that;
# otherwise sort by unscaled MAE to see the spread
order = np.argsort(pm)
print("per-client unscaled MAE, sorted:")
for i in order:
    print(f"  client {i:2d}: MAE={pm[i]:7.3f}  n={int(ns[i])}")
print(f"\nspread: min={pm.min():.3f}  max={pm.max():.3f}  ratio={pm.max()/pm.min():.1f}×")

In [ ]:
import numpy as np
r = results_fedavg[(0.01, 42)]
pm = np.array(r["per_user"]["metrics"]["unscaled_mae"])
ns = np.array(r["per_user"]["num_samples"])

w_all = np.sum(pm*ns)/ns.sum()
keep = np.arange(len(pm)) != 14
w_no14 = np.sum(pm[keep]*ns[keep])/ns[keep].sum()
print(f"weighted unscaled MAE, all clients : {w_all:.3f}")
print(f"weighted unscaled MAE, drop client 14: {w_no14:.3f}")
print(f"client 14 alone: MAE={pm[14]:.1f}  n={int(ns[14])}  "
      f"share of total error={100*pm[14]*ns[14]/np.sum(pm*ns):.1f}%")

# and check the n=176 block
print(f"\nclients at exactly n=176: {np.sum(ns==176)}  "
      f"(Dirichlet shouldn't produce identical counts — likely a min-size floor)")

## Per-client MAE distribution by α  ← the heterogeneity signal

Under **client-local** evaluation this is where heterogeneity surfaces: at low α,
clients trained on narrow regimes (and on very different sample counts) should
show a **wider spread** of per-client MAE. Under the **global** pool every client
is tested identically, so this spread reflects only model differences.

In [ ]:
pu_by_alpha = {a: r.get("per_user", {}).get("metrics", {}).get("unscaled_mae", [])
               for a, r in results_fedavg.items()}
avail = [a for a in ALPHAS if pu_by_alpha.get(a)]

if avail:
    fig, ax = plt.subplots(figsize=(10, 5))
    data = [np.asarray(pu_by_alpha[a], float) for a in avail]
    bp = ax.boxplot(data, labels=[str(a) for a in avail], showmeans=True,
                    patch_artist=True)
    for box in bp["boxes"]:
        box.set(facecolor=uc2.COLORS["FedAvg"], alpha=0.5)
    # overlay raw points (jittered) so small-N alphas are honest
    for i, d in enumerate(data, start=1):
        x = np.random.normal(i, 0.04, size=len(d))
        ax.plot(x, d, ".", color="k", alpha=0.4, ms=4)
    ax.set_xlabel("α (lower = more heterogeneous)")
    ax.set_ylabel("Per-client unscaled MAE (MB)")
    ax.set_title(f"FedAvg — per-client MAE spread by α  [{PARTITION_VARIANT}]")
    plt.tight_layout(); plt.show()

    print(f"{'α':<8}{'median':>10}{'mean':>10}{'std':>10}{'CV':>8}{'max/med':>10}{'n':>5}")
    for a in avail:
        d = np.asarray(pu_by_alpha[a], float)
        med = np.median(d)
        print(f"{a:<8}{med:>10.3f}{d.mean():>10.3f}{d.std():>10.3f}"
              f"{d.std()/d.mean():>8.3f}{(d.max()/med if med else float('nan')):>10.2f}{len(d):>5}")
else:
    print("No per-user metrics found — run the training cell first.")

## Global metric vs α

In [ ]:
def best_mae(r):
    maes = [m.get("unscaled_mae", float("inf")) for m in r["metrics"]["glob_test_metric"]]
    return min(maes) if maes else float("nan")

print(f"{'α':<8}{'best MAE':>12}{'last MAE':>12}{'best round':>12}{'per-client σ':>14}")
for a in sorted(results_fedavg):
    r = results_fedavg[a]
    maes = [m.get("unscaled_mae", float("inf")) for m in r["metrics"]["glob_test_metric"]]
    pu = pu_by_alpha.get(a, [])
    print(f"{a:<8}{min(maes):>12.4f}{maes[-1]:>12.4f}{int(np.argmin(maes)):>12}"
          f"{(np.std(pu) if pu else 0):>14.4f}")

## Gap to Centralized  (run after notebook 01b on the SAME variant)

The honest α-sensitivity readout: `MAE(FedAvg, α) − MAE(Centralized, α)`. Centralized
is the α-independent ceiling. If the gap widens at low α, FedAvg is genuinely hurt
by heterogeneity even when its raw MAE looks flat.

Requires Centralized results under the SAME variant's results folder
(`results/<newpart|newpart_global>/centralized/...`). If absent, this cell just
skips the gap and shows FedAvg alone.

In [ ]:
def _load_variant_result(algorithm, alpha):
    p = os.path.join(uc2.RESULTS, _NP_RESULTS, algorithm.lower(),
                     f"alpha_{alpha}", MODEL, "rep_0", "full_results.pkl")
    if not os.path.exists(p):
        return None
    with open(p, "rb") as f:
        return pickle.load(f)

cent = {a: _load_variant_result("Centralized", a) for a in ALPHAS}
have_cent = any(v is not None for v in cent.values())

fa_mae  = {a: best_mae(results_fedavg[a]) for a in results_fedavg}
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(list(fa_mae), [fa_mae[a] for a in fa_mae], "o-",
        color=uc2.COLORS["FedAvg"], lw=2, label="FedAvg")
if have_cent:
    cm = {a: best_mae(cent[a]) for a in cent if cent[a]}
    ax.plot(list(cm), [cm[a] for a in cm], "s--",
            color=uc2.COLORS["Centralized"], lw=2, label="Centralized")
ax.set_xscale("log"); ax.set_xlabel("α (log)"); ax.set_ylabel("Best unscaled MAE (MB)")
ax.set_title(f"FedAvg vs Centralized — {PARTITION_VARIANT}")
ax.legend(); plt.tight_layout(); plt.show()

if have_cent:
    print(f"{'α':<8}{'FedAvg':>10}{'Central':>10}{'gap':>10}{'gap %':>9}")
    for a in sorted(fa_mae):
        if cent.get(a):
            c = best_mae(cent[a]); g = fa_mae[a] - c
            print(f"{a:<8}{fa_mae[a]:>10.3f}{c:>10.3f}{g:>10.3f}"
                  f"{(100*g/c if c else float('nan')):>8.1f}%")
else:
    print("[info] No Centralized results for this variant yet — "
          "run notebook 01 retrained on the same PARTITION_VARIANT, then re-run this cell.")

In [ ]:
import numpy as np
_args = uc2.make_args("FedAvg", 0.01, seed=0, **OVERRIDES)
from utils.model_utils import read_data, read_user_data
from FLAlgorithms.users.useravg import UserAVG
_data = read_data(_args.dataset, _args.dataset_path)
_cid, _tr, _te = read_user_data(0, _data, dataset=_args.dataset, model=_args.model)
_u = UserAVG(_args, _cid, [__import__('torch').load(
    __import__('os').path.join(_args.result_path,"models",_args.dataset,"server.pt"),
    map_location=_args.device, weights_only=False), _args.model], _tr, _te, use_adam=False)

nf = next(iter(_u.testloaderfull))[0].shape[-1]
print("n_feat =", nf)
for probe_in in [np.zeros((1, nf)), np.zeros((3, nf)), np.array([0.5]), np.array([0.5, 0.5, 0.5])]:
    try:
        out = _u.inverse_preprocessing(_u.dataset_path_prefix, "train", probe_in, axis=0)
        print(f"in {np.shape(probe_in)} -> out shape {np.shape(out)}  vals {np.ravel(out)[:3]}")
    except Exception as e:
        print(f"in {np.shape(probe_in)} -> {type(e).__name__}: {e}")

In [ ]:
import pickle, os, glob
root = os.path.join(uc2.DATA_PART, "new_partitions", "lookback_60", "steps_1")
hits = glob.glob(os.path.join(root, "u20-alpha*-ratio1", "train_scaler.pkl"))

stats = {}
for p in sorted(hits):
    alpha = p.split("alpha")[1].split("-ratio")[0]
    s = pickle.load(open(p, "rb"))
    stats[alpha] = (round(float(s.data_min_[0]), 5), round(float(s.data_max_[0]), 5))
    print(f"α={alpha}: target-col min={stats[alpha][0]}  max={stats[alpha][1]}")
print("\n", "ONE shared scaler" if len(set(stats.values()))==1
      else f"{len(set(stats.values()))} DISTINCT scalers")

## Communication cost

In [ ]:
for a, r in sorted(results_fedavg.items()):
    n = r["n_rounds"]
    print(f"α={a}: {n} rounds → {uc2.comm_cost_fedavg(n, n_users_per_round=OVERRIDES['num_users']):.1f} MB")

## Notes

- **First run per variant:** `RETRAIN=True`. Flip `PARTITION_VARIANT` to `"global"`
  and re-run the whole notebook to produce the Option-B results.
- **Baseline:** retrain Centralized (notebook 01) under the *same* variant so the
  gap-to-Centralized cell has matching test sets. Centralized's pooled model is
  α-invariant, so this is a cheap re-evaluation, not a real retrain.
- **Reading the results on this dataset:** quantity skew is the dominant
  heterogeneity axis here (label skew is mild), so the per-client spread plot and
  the gap-to-Centralized are more informative than the global-mean MAE curve.


In [ ]:
import pickle, os, numpy as np
p = os.path.join(uc2.RESULTS,"newpart","fedavg","alpha_0.01","lstm","rep_0","full_results.pkl")
r = pickle.load(open(p,"rb"))
sc = np.array([m["mae"] for m in r["metrics"]["glob_test_metric"]])      # SCALED
un = np.array([m["unscaled_mae"] for m in r["metrics"]["glob_test_metric"]])
i = int(np.argmin(sc))
print(f"SCALED: best={sc[i]:.5f} @round {i}  (first {sc[0]:.5f} -> last {sc[-1]:.5f})  margin={len(sc)-1-i}")
print(f"UNSCALED (what the notebook shows): best={un.min():.4f}  range over run={un.max()-un.min():.5f}")

In [ ]:
bi = int(np.argmin([m.get("mae", float("inf")) for m in glob]))
b = glob[bi]
print(f"  best round {bi}: scaled MAE={b['mae']:.5f}")
pu = result["per_user"]["metrics"].get("mae", [])   # scaled per-client too

In [ ]:
import pickle, os, numpy as np
for a in [0.01, 0.1, 10.0]:
    r = pickle.load(open(os.path.join(uc2.RESULTS,"newpart","fedavg",f"alpha_{a}","lstm","rep_0","full_results.pkl"),"rb"))
    pm = np.array(r["per_user"]["metrics"]["mae"])      # scaled per-client MAE
    ns = np.array(r["per_user"]["num_samples"])
    order = np.argsort(ns)                                # small -> large clients
    print(f"\nα={a}:")
    print(f"  unweighted per-client MAE: mean={pm.mean():.4f}  median={np.median(pm):.4f}")
    print(f"  sample-weighted MAE       : {np.sum(pm*ns)/np.sum(ns):.4f}   <- what the plot shows")
    print(f"  smallest-5 clients MAE: {pm[order[:5]].mean():.4f}   n≈{ns[order[:5]].mean():.0f}")
    print(f"  largest-5  clients MAE: {pm[order[-5:]].mean():.4f}   n≈{ns[order[-5:]].mean():.0f}")

In [ ]:
import pickle, os, numpy as np
import matplotlib.pyplot as plt
uc2.setup_plot_style()

alphas = sorted(results_fedavg.keys())
weighted, macro, median = [], [], []
for a in alphas:
    r = pickle.load(open(os.path.join(uc2.RESULTS,"newpart","fedavg",
                         f"alpha_{a}","lstm","rep_0","full_results.pkl"),"rb"))
    pm = np.array(r["per_user"]["metrics"]["mae"])
    ns = np.array(r["per_user"]["num_samples"])
    weighted.append(np.sum(pm*ns)/np.sum(ns))   # what the convergence plot showed
    macro.append(pm.mean())                      # unweighted: every client counts once
    median.append(np.median(pm))

x = range(len(alphas))
fig, ax = plt.subplots(figsize=(9,5))
ax.plot(x, weighted, "o-", lw=2, label="sample-weighted (big clients dominate)")
ax.plot(x, macro,    "s-", lw=2, label="unweighted macro-mean")
ax.plot(x, median,   "^--", lw=2, label="median client")
ax.set_xticks(x); ax.set_xticklabels(alphas)
ax.set_xlabel("Dirichlet α"); ax.set_ylabel("Scaled MAE (final model)")
ax.set_title("FedAvg — weighting changes the α story (client-local)")
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9,5))
data, labels = [], []
for a in alphas:
    r = pickle.load(open(os.path.join(uc2.RESULTS,"newpart","fedavg",
                         f"alpha_{a}","lstm","rep_0","full_results.pkl"),"rb"))
    data.append(np.array(r["per_user"]["metrics"]["mae"]))   # one point per client
    labels.append(f"α={a}")
ax.boxplot(data, labels=labels, showmeans=True)
ax.set_ylabel("Scaled MAE (per client)"); ax.set_xlabel("Dirichlet α")
ax.set_title("FedAvg — per-client MAE distribution (client-local)")
plt.tight_layout(); plt.show()

In [ ]:
import glob, os, pickle, re
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# 0. PER-ROUND PAYLOAD (MB) — measure from the model, don't hardcode
# ============================================================
# If you have the model object in the notebook (e.g. `server.model` or the
# `create_model` result), measure it. Otherwise fall back to nominal + WARN.
def bytes_of(params):
    return sum(p.numel() for p in params) * 4  # float32

try:
    _full_params   = list(server.model.parameters())            # adjust var name
    MODEL_MB       = bytes_of(_full_params) / 1e6
    # partial shares only the encoder/shared layers:
    SHARED_MB      = bytes_of(server.model.get_shared_parameters()) / 1e6
    # FedGen also broadcasts the generator each round (shared part):
    try:    GEN_MB = bytes_of(server.generative_model.parameters()) / 1e6
    except Exception: GEN_MB = 0.0
    print(f"[measured] full model={MODEL_MB:.3f} MB  shared={SHARED_MB:.3f} MB  gen={GEN_MB:.3f} MB")
except Exception as e:
    print(f"[WARN] could not measure model ({e}); using NOMINAL constants — flag in thesis")
    MODEL_MB, SHARED_MB, GEN_MB = 3.7, 0.4, 1.0   # replace with your real nominal values

# per-round upload+download cost (×2 = client→server + server→client)
PAYLOAD = {
    "fedavg":         2 * MODEL_MB,
    "fedavg-partial": 2 * SHARED_MB,
    "fedgen":         2 * MODEL_MB + GEN_MB,      # full model + generator broadcast
    "fedgen-partial": 2 * SHARED_MB + GEN_MB,
}

# ============================================================
# 1. LOAD per-round MACRO MAE for each method (client_local / newpart)
# ============================================================
ROOT = uc2.RESULTS  # or hardcode the newpart root
METHODS = ["fedavg", "fedgen", "fedgen-partial"]   # add "fedavg-partial" if you ran it
COLORS  = {"fedavg":"tab:blue","fedgen":"tab:orange",
           "fedgen-partial":"tab:green","fedavg-partial":"tab:red"}
ALPHA_TO_SHOW = 0.5   # pick one α for the per-round figures; sweep separately if you like

def load_macro_curve(method, alpha):
    """Return per-round MACRO MAE (unweighted mean over clients is not stored
    per-round; glob_test_metric is sample-weighted. So we plot the stored
    per-round 'mae' (weighted) for the CURVE SHAPE, and mark the final macro
    point from per_user). See note below."""
    p = os.path.join(ROOT,"newpart",method,f"alpha_{alpha}","lstm","rep_0","full_results.pkl")
    if not os.path.exists(p): return None
    r = pickle.load(open(p,"rb"))
    weighted = np.array([m["mae"] for m in r["metrics"]["glob_test_metric"]])
    pm = np.array(r["per_user"]["metrics"]["mae"]); macro_final = pm.mean()
    return dict(weighted=weighted, macro_final=macro_final, n=len(weighted))

# ============================================================
# FIGURE A — Pareto frontier: MAE vs cumulative COMMUNICATION (MB)
# ============================================================
uc2.setup_plot_style()
figA, axA = plt.subplots(figsize=(9,5.5))
for method in METHODS:
    c = load_macro_curve(method, ALPHA_TO_SHOW)
    if c is None: continue
    rounds = np.arange(1, c["n"]+1)
    cum_mb = rounds * PAYLOAD[method]            # cumulative MB transmitted
    axA.plot(cum_mb, c["weighted"], "-", color=COLORS[method], lw=2, label=method)
    # mark best-MAE point (the efficient operating point)
    bi = int(np.argmin(c["weighted"]))
    axA.scatter(cum_mb[bi], c["weighted"][bi], color=COLORS[method],
                s=90, zorder=5, edgecolor="black")
axA.set_xscale("log")
axA.set_xlabel("Cumulative communication (MB, log scale)")
axA.set_ylabel("Scaled MAE")
axA.set_title(f"UC2 — accuracy vs communication cost (α={ALPHA_TO_SHOW}, client-local)")
axA.legend(); axA.grid(alpha=0.3, which="both")
plt.tight_layout(); plt.show()

# ============================================================
# FIGURE B — Convergence speed: MAE vs ROUND, methods overlaid
# ============================================================
figB, axB = plt.subplots(figsize=(9,5.5))
for method in METHODS:
    c = load_macro_curve(method, ALPHA_TO_SHOW)
    if c is None: continue
    rounds = np.arange(1, c["n"]+1)
    axB.plot(rounds, c["weighted"], "-", color=COLORS[method], lw=2, label=method)
    bi = int(np.argmin(c["weighted"]))
    axB.scatter(bi+1, c["weighted"][bi], color=COLORS[method],
                s=90, zorder=5, edgecolor="black",
                label=f"{method} best @r{bi+1}")
axB.set_xlabel("Communication round")
axB.set_ylabel("Scaled MAE")
axB.set_title(f"UC2 — convergence speed (α={ALPHA_TO_SHOW}, client-local)")
axB.legend(fontsize=8); axB.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# ── Weighted vs macro vs median MAE across α  (the inversion is a weighting artifact) ──
# Uses the trustworthy SCALED per-client mae + num_samples already in results_fedavg.
# weighted = Σ(mae_k·n_k)/Σn_k   |   macro = mean(mae_k)   |   median = median(mae_k)

N_LARGE = 5  # size of the "largest"/"smallest" client groups in panel B

def _aggts(r):
    pu  = r.get("per_user", {})
    mae = np.asarray(pu.get("metrics", {}).get("mae", []), float)   # SCALED, correct
    ns  = np.asarray(pu.get("num_samples", []), float)
    if mae.size == 0 or ns.size != mae.size:
        return None
    o = np.argsort(ns)
    return dict(
        weighted=float(np.sum(mae*ns)/np.sum(ns)),
        macro=float(mae.mean()),
        median=float(np.median(mae)),
        large=float(mae[o[-N_LARGE:]].mean()),
        small=float(mae[o[:N_LARGE]].mean()),
    )

rows = {a: _aggts(results_fedavg[a]) for a in sorted(results_fedavg)}
rows = {a: v for a, v in rows.items() if v is not None}

if not rows:
    print("No per-client scaled mae / num_samples found — run the training cell first.")
else:
    alphas = list(rows)
    x = np.arange(len(alphas))
    W  = [rows[a]["weighted"] for a in alphas]
    MA = [rows[a]["macro"]    for a in alphas]
    ME = [rows[a]["median"]   for a in alphas]
    L  = [rows[a]["large"]    for a in alphas]
    S  = [rows[a]["small"]    for a in alphas]

    uc2.setup_plot_style()
    fig, (axA, axB) = plt.subplots(1, 2, figsize=(13, 5), gridspec_kw=dict(width_ratios=[1.35, 1.0]))

    # Panel A — same data, three aggregations; only the weighted curve inverts
    axA.plot(x, W,  "o-",  color=uc2.COLORS["FedAvg"], lw=2.6, ms=9, zorder=5,
             label="Sample-weighted (plotted curve)")
    axA.plot(x, MA, "s--", color="#2E86C1", lw=2.2, ms=8, label="Unweighted mean across clients")
    axA.plot(x, ME, "D--", color="#27AE60", lw=2.2, ms=8, label="Median across clients")
    axA.set_xticks(x); axA.set_xticklabels([f"α={a}" for a in alphas])
    axA.set_xlabel("α  (left = more heterogeneous)")
    axA.set_ylabel("Scaled MAE  (lower = better)")
    axA.set_title("A.  Inversion appears only in the sample-weighted curve")
    axA.legend()

    # Panel B — why: at low α the largest clients are the easiest ones
    axB.plot(x, L, "o-", color="#E67E22", lw=2.4, ms=8, label=f"Largest-{N_LARGE} clients")
    axB.plot(x, S, "^-", color="#8E44AD", lw=2.4, ms=8, label=f"Smallest-{N_LARGE} clients")
    for i in range(len(x)):
        if L[i] < S[i]:                       # big clients EASIER than small -> shade
            axB.axvspan(x[i]-0.42, x[i]+0.42, color="#E67E22", alpha=0.10, zorder=0)
    axB.set_xticks(x); axB.set_xticklabels([f"α={a}" for a in alphas])
    axB.set_xlabel("α  (left = more heterogeneous)")
    axB.set_ylabel("Scaled MAE")
    axB.set_title(f"B.  Dominant clients sit in the easy regime at low α  [{PARTITION_VARIANT}]")
    axB.legend()

    plt.tight_layout(); plt.show()

    print(f"{'α':<8}{'weighted':>10}{'macro':>10}{'median':>10}{'large5':>10}{'small5':>10}")
    for a in alphas:
        v = rows[a]
        print(f"{a:<8}{v['weighted']:>10.4f}{v['macro']:>10.4f}{v['median']:>10.4f}"
              f"{v['large']:>10.4f}{v['small']:>10.4f}")

---
# R2 — Local-epochs sweep: client drift in FedAvg vs FedGen

The main runs use `local_epochs=20`. **Client drift** grows with local steps/round: more local
epochs on heterogeneous data → local models diverge → the average degrades. The clean way to
*name* that mechanism is to vary `local_epochs` and show the **trend** for both methods, not a
single point.

This sweep re-runs **FedAvg and FedGen** at `local_epochs ∈ {1, 5, 10}` (you already have both at
20) across all α, then plots error vs local_epochs. Expectation: FedAvg worsens as local epochs
rise (drift); FedGen stays flat (its KD regularization pulls clients to consensus).

**Nothing is overwritten:** each config saves to its own folder — `FedAvg-le{E}` →
`results/newpart/fedavg-le{E}/…`, `FedGen-le{E}` → `…/fedgen-le{E}/…`. Your `fedavg/`, `fedgen/`,
`fedgen-partial/`, `centralized/` (all le=20) are read-only here. Resumable (skip-if-exists);
seed 0 only (one seed is enough for the local-epochs trend).

> Requires the wiring cell to have run with `PARTITION_VARIANT='client_local'` and the le=20
> baselines (`results/newpart/fedavg/`, `results/newpart/fedgen/`) to exist.

In [ ]:
# R2.2 — sweep runner: FedAvg AND FedGen over local_epochs (resumable; nothing overwritten)
import time, gc

SANITY_METHODS      = ["FedAvg", "FedGen"]   # baseline + proposed method
SANITY_LOCAL_EPOCHS = [1, 5, 10]             # 20 = your existing baselines (fedavg/, fedgen/)
SANITY_SEEDS        = [0]                     # one seed is enough for the local-epochs trend
SANITY_ALPHAS       = ALPHAS                  # all six

assert _NEWPART_ON["flag"] and PARTITION_VARIANT == "client_local", \
    "Run the wiring cell with PARTITION_VARIANT='client_local' before this sweep."
assert uc2.make_args is _patched_make_args, "uc2.make_args not patched -- re-run the wiring cell."

def _sweep_algo(method, le):                     # encode method+local_epochs -> distinct subdir
    return f"{method}-le{le}"
def _sweep_pkl(method, le, seed, alpha):
    return os.path.join(uc2.RESULTS, _NP_RESULTS, _sweep_algo(method, le).lower(),
                        f"alpha_{alpha}", MODEL, f"rep_{seed}", "full_results.pkl")

plan = [(m, le, s, a) for s in SANITY_SEEDS for m in SANITY_METHODS
                      for le in SANITY_LOCAL_EPOCHS for a in SANITY_ALPHAS]
todo = [x for x in plan if not os.path.exists(_sweep_pkl(*x))]
def _rt(m, le): return 300 * (le * (0.9 if m == "FedGen" else 0.28) + 0.8)   # FedGen ~3x slower
est = sum(_rt(m, le) for m, le, s, a in todo) / 60
print(f"local-epochs sweep -- {len(plan)} runs ({len(todo)} to run, {len(plan)-len(todo)} cached)")
print(f"  methods={SANITY_METHODS}  local_epochs={SANITY_LOCAL_EPOCHS}  seeds={SANITY_SEEDS}  alphas={SANITY_ALPHAS}")
print(f"  upper-bound estimate: ~{est:.0f} min (~{est/60:.1f} h); resumable\n")

t0 = time.time()
for i, (m, le, seed, alpha) in enumerate(plan, 1):
    tag = f"{m}-le{le} seed={seed} a={alpha}"
    if os.path.exists(_sweep_pkl(m, le, seed, alpha)):
        print(f"[{i:>2}/{len(plan)}] {tag}: cached"); continue
    print(f"[{i:>2}/{len(plan)}] {tag}: training...  (elapsed {(time.time()-t0)/60:.0f} min)")
    try:
        server, _ = uc2.run_experiment(algorithm=_sweep_algo(m, le), alpha=alpha,
                                       seed=seed, **{**OVERRIDES, "local_epochs": le})
        del server; gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception as e:
        import traceback; traceback.print_exc(); print(f"  [ERROR] {tag}: {e}")
print(f"\n[done] sweep finished in {(time.time()-t0)/60:.1f} min")


## Verdict — error vs local_epochs (the client-drift trend)

Best scaled MSE per method at each `local_epochs` (1/5/10 from the sweep, 20 from your existing
baselines). Read down a method's row: if FedAvg's MSE **rises** with local_epochs while FedGen's
stays low, that is client drift — and the differential is FedGen's contribution. The figure below
makes it visual, in both scaled MSE and real-units MAE.

In [ ]:
# R2.3 — table: best error by local_epochs, per method
def _best(r, key):
    v = [m[key] for m in r["metrics"]["glob_test_metric"]]
    bi = int(np.argmin(v)); return v[bi], bi
def _pkl(method_dir, seed, alpha):
    p = os.path.join(uc2.RESULTS, _NP_RESULTS, method_dir, f"alpha_{alpha}", MODEL, f"rep_{seed}", "full_results.pkl")
    return pickle.load(open(p, "rb")) if os.path.exists(p) else None
def _dir(method, le):
    return method.lower() if le == 20 else _sweep_algo(method, le).lower()

LE_ALL = sorted(set(SANITY_LOCAL_EPOCHS + [20]))
seed0 = SANITY_SEEDS[0]
print("best scaled MSE by local_epochs  (le=20 = existing baseline):")
print(f"{'a':<7}{'method':<9}" + "".join(f"le={le:<8}" for le in LE_ALL))
print("-" * (16 + 11 * len(LE_ALL)))
for alpha in SANITY_ALPHAS:
    for method in SANITY_METHODS:
        row = f"{alpha:<7}{method:<9}"
        for le in LE_ALL:
            r = _pkl(_dir(method, le), seed0, alpha)
            row += (f"{_best(r, 'mse')[0]:<11.4f}" if r else f"{'--':<11}")
        print(row)
    print()
print("If FedAvg's row climbs with local_epochs and FedGen's stays flat -> differential client drift.")


In [ ]:
# R2.4 — CLIENT-DRIFT FIGURE: error vs local_epochs, FedAvg vs FedGen
LE_ALL = sorted(set(SANITY_LOCAL_EPOCHS + [20]))
seed0 = SANITY_SEEDS[0]
PLOT_ALPHA = 1.0
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, key, ylab in [(axes[0], "mse", "best scaled MSE"),
                      (axes[1], "unscaled_mae", "best real MAE (MB)")]:
    for method, c in [("FedAvg", "tab:red"), ("FedGen", "tab:green")]:
        xs, ys = [], []
        for le in LE_ALL:
            r = _pkl(_dir(method, le), seed0, PLOT_ALPHA)
            if r:
                xs.append(le); ys.append(_best(r, key)[0])
        if xs:
            ax.plot(xs, ys, "o-", color=c, lw=2, ms=7, label=method)
    ax.set_xlabel("local epochs per round"); ax.set_ylabel(ylab)
    ax.set_xticks(LE_ALL); ax.grid(alpha=.3); ax.legend()
fig.suptitle(f"Client drift: error vs local epochs  [client_local, a={PLOT_ALPHA}]", fontweight="bold")
plt.tight_layout(); plt.show()
